# Assignment 5
## Data Preprocessing

We will be using a dataframe created from *Income Dirty Data.csv*. Download the file from D2L. 

1. Import the following modules
    - `pandas`
    - `numpy`
    - `preprocessing` from `sklearn` (for bonus question)
    - `KNNImputer` from `sklearn.impute` (for bonus question)
2. Create your dataframe from the file using `pandas`

In [2]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.impute import KNNImputer

df = pd.read_csv("Income Dirty Data.csv")

df.head()

,ID,sex,age,income,tax_15_pct
0,1,Women,21,147168.0,22075.20
1,2,Female,29,119595.0,17939.25
2,3,Female,56,87770.0,13165.50
3,4,NaN,21,54259.0,8138.85
4,5,Male,28,NaN,160230.00


3. Calculate and display the following information
    - Total number of NaN values for **each column**
    - Percentage of NaN values in the dataset 
    - Number of rows *without* any NaN values

In [3]:
print("NaN values for each column:")
print(df.isna().sum())

total_nan = df.isna().sum().sum()
total_cells = df.shape[0] * df.shape[1]

nan_percentage = (total_nan / total_cells) * 100

print("\nPercentage of NaN values in the dataset:")
print(nan_percentage)

complete_rows = df.dropna().shape[0]

print("\nNumber of rows without any NaN values:")
print(complete_rows)

NaN values for each column:
ID              0
sex            88
age             0
income        109
tax_15_pct     93
dtype: int64

Percentage of NaN values in the dataset:
5.800000000000001

Number of rows without any NaN values:
733


Besides missing values (NaN), the dataset contains errors. We have the following rules to check:

* All employees are adults (18+ years old)
* All employees pay 15% of their income for the tax
* All employees make money; no income should be <= 0 
---
4. Calculate and display the percentage of the data that does **NOT** violate any **one** of the rules.

In [4]:
rule_age = df["age"] >= 18

rule_tax = np.isclose(
    df["tax_15_pct"],
    df["income"] * 0.15,
    equal_nan=False
)

rule_income = df["income"] > 0

valid_rows = rule_age & rule_tax & rule_income

valid_percentage = valid_rows.mean() * 100

print("Percentage of data that does not violate any rule:")
print(valid_percentage)

Percentage of data that does not violate any rule:
59.8


Now that we have determined the number of erroneous datapoints in our set, let's work on correcting it as best we can.

5. Replace non *Female*/*Male* values in the **Sex** column with either *Female* or *Male* (e.g., Women --> Female)

In [5]:
df["sex"] = df["sex"].replace({
    "Women": "Female",
    "Woman": "Female",
    "women": "Female",
    "woman": "Female",
    "Men": "Male",
    "Man": "Male",
    "men": "Male",
    "man": "Male"
})

print(df["sex"].unique())

<StringArray>
['Female', nan, 'Male']
Length: 3, dtype: str


6. Replace non-positive **Age** values with NaN (`numpy.NaN`)
7. Replace non-positive **Income** values with NaN (`numpy.NaN`)
8. Replace non-positive **Tax (15%)** values with NaN (`numpy.NaN`)

In [6]:
df.loc[df["age"] <= 0, "age"] = np.nan
df.loc[df["income"] <= 0, "income"] = np.nan
df.loc[df["tax_15_pct"] <= 0, "tax_15_pct"] = np.nan

df.head()

,ID,sex,age,income,tax_15_pct
0,1,Female,21.0,147168.0,22075.20
1,2,Female,29.0,119595.0,17939.25
2,3,Female,56.0,87770.0,13165.50
3,4,NaN,21.0,54259.0,8138.85
4,5,Male,28.0,NaN,160230.00


The following question is a bonus (+10) question, but I'd encourage you to give it a try!

9. Use machine learning (`KNNImputer`) to impute all missing values (replaces NaN values with the most predicted values)
    - Will need to use a scaler and convert the values in the **Sex** column to a numeric value for algorithm to work properly
    - Show some of the data prior to imputing, and after imputing

In [9]:
knn_df = df.copy()

print("Data before imputation:")
display(knn_df.head(10))

knn_df["sex"] = knn_df["sex"].map({
    "Female": 0,
    "Male": 1
})

print("Data after converting sex to numeric:")
display(knn_df.head(10))

impute_columns = ["sex", "age", "income", "tax_15_pct"]

knn_data = knn_df[impute_columns].copy()

knn_data.head()

scaler = preprocessing.StandardScaler()

scaled_data = scaler.fit_transform(knn_data)

imputer = KNNImputer(n_neighbors=5)

imputed_scaled_data = imputer.fit_transform(scaled_data)

imputed_data = scaler.inverse_transform(imputed_scaled_data)

imputed_df = pd.DataFrame(
    imputed_data,
    columns=impute_columns,
    index=knn_df.index
)

imputed_df["sex"] = imputed_df["sex"].round().map({
    0: "Female",
    1: "Male"
})

imputed_df.insert(0, "ID", knn_df["ID"].values)

print("Data after KNN imputation:")
display(imputed_df.head(10))

Data before imputation:


,ID,sex,age,income,tax_15_pct
0,1,Female,21.0,147168.0,22075.20
1,2,Female,29.0,119595.0,17939.25
2,3,Female,56.0,87770.0,13165.50
3,4,NaN,21.0,54259.0,8138.85
4,5,Male,28.0,NaN,160230.00
5,6,Female,NaN,128326.0,19248.90
6,7,Female,NaN,NaN,NaN
7,8,Female,24.0,NaN,11820.60
8,9,Female,38.0,149473.0,22420.95
9,10,Male,48.0,113663.0,1136630.00


Data after converting sex to numeric:


,ID,sex,age,income,tax_15_pct
0,1,0.0,21.0,147168.0,22075.20
1,2,0.0,29.0,119595.0,17939.25
2,3,0.0,56.0,87770.0,13165.50
3,4,NaN,21.0,54259.0,8138.85
4,5,1.0,28.0,NaN,160230.00
5,6,0.0,NaN,128326.0,19248.90
6,7,0.0,NaN,NaN,NaN
7,8,0.0,24.0,NaN,11820.60
8,9,0.0,38.0,149473.0,22420.95
9,10,1.0,48.0,113663.0,1136630.00


Data after KNN imputation:


,ID,sex,age,income,tax_15_pct
0,1,Female,21.0,147168.0,22075.20
1,2,Female,29.0,119595.0,17939.25
2,3,Female,56.0,87770.0,13165.50
3,4,Male,21.0,54259.0,8138.85
4,5,Male,28.0,87077.0,160230.00
5,6,Female,36.2,128326.0,19248.90
6,7,Female,40.8,76111.4,14991.18
7,8,Female,24.0,110944.4,11820.60
8,9,Female,38.0,149473.0,22420.95
9,10,Male,48.0,113663.0,1136630.00


10. In the empty `Markdown` cell below, explain why it is important to clean a dataset before calculating analytics about the data

### Why Data Cleaning Is Important

Data cleaning is important because inaccurate, incomplete, or inconsistent data can lead to incorrect analytical results. Missing values can cause calculations to be incomplete, while invalid values can distort averages, relationships, and other statistics. For example, an employee with a negative income or an age below zero would not represent a realistic observation and could negatively affect an analysis. Standardizing values, identifying invalid data, and handling missing values makes the dataset more consistent and reliable. Cleaning the data before performing analytics helps ensure that the conclusions and predictions made from the data are based on accurate and meaningful information.

### Submission to D2L Dropbox
- Submit this `Jupyter` file to D2L, renamed as **Last_First_Assignment5.ipynb** 
    - Replace '**Last**' and '**First**' with your first and last name

- Include the link to your GitHub repository (the URL of your repo page, for
   example `https://github.com/yourname/csci-4047-work`).
### How to add my code to GitHub?

1. Stage your file (this tells Git which changes to include):

      `git add Last_First_Assignment5.ipynb`

   To stage everything (you might not want to stage everything though) in the folder instead, use `git add .`

3. Commit your changes (this saves a snapshot with a message):

       git commit -m "Add Assignment 5"

   The text in quotes is your commit message. Make it describe what you
   did.

4. Push your commit up to GitHub:

       git push -u origin main

   The `-u origin main` part is only needed the first push. After that,
   `git push` alone is enough.

**What each command does, briefly**

- `git add` picks which files to include in the next save.
- `git commit` saves a snapshot of those files on your computer, with a
  message describing the change.
- `git push` uploads your saved commits to GitHub so they appear online.